# PyTorch Tutorial - Solution Notebook
This notebook runs through all examples to verify they work correctly.

In [1]:
# Test imports and setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")
print(f"Python version: {torch.version.cuda if torch.cuda.is_available() else 'CUDA not available'}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.1.0
Python version: CUDA not available
CUDA available: False


In [2]:
# Test tensor creation
data = [[1, 2], [3, 4]]
tensor_from_list = torch.tensor(data)
print("From list:")
print(tensor_from_list)
print(f"Shape: {tensor_from_list.shape}")
print(f"Data type: {tensor_from_list.dtype}")
print()

From list:
tensor([[1, 2],
        [3, 4]])
Shape: torch.Size([2, 2])
Data type: torch.int64



In [3]:
# Test tensor creation with specific shapes
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
random = torch.randn(2, 3)

print("Zeros tensor:")
print(zeros)
print("\nOnes tensor:")
print(ones)
print("\nRandom tensor:")
print(random)

Zeros tensor:
tensor([[0., 0., 0.],
        [0., 0., 0.]])

Ones tensor:
tensor([[1., 1., 1.],
        [1., 1., 1.]])

Random tensor:
tensor([[-1.2281,  0.3292, -1.5256],
        [ 1.0015, -0.4839, -1.0020]])


In [4]:
# Test tensor properties
tensor = torch.randn(3, 4, 5)

print(f"Tensor shape: {tensor.shape}")
print(f"Tensor size: {tensor.size()}")
print(f"Number of dimensions: {tensor.ndim}")
print(f"Data type: {tensor.dtype}")
print(f"Device: {tensor.device}")
print(f"Total number of elements: {tensor.numel()}")

Tensor shape: torch.Size([3, 4, 5])
Tensor size: torch.Size([3, 4, 5])
Number of dimensions: 3
Data type: torch.float32
Device: cpu
Total number of elements: 60


In [5]:
# Test autograd
x = torch.tensor([2.0], requires_grad=True)
print(f"x = {x}")
print(f"x.requires_grad = {x.requires_grad}")
print()

y = x**2 + 3*x + 1
print(f"y = x^2 + 3x + 1 = {y}")
print()

y.backward()
print(f"dy/dx = {x.grad}")
print(f"Expected gradient: 2*{x.item()} + 3 = {2*x.item() + 3}")

x = tensor([2.], requires_grad=True)
x.requires_grad = True

y = x^2 + 3x + 1 = tensor([11.], grad_fn=<AddBackward0>)

dy/dx = tensor([7.])
Expected gradient: 2*2.0 + 3 = 7.0


In [6]:
# Test neural network creation
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

model = SimpleNet(input_size=4, hidden_size=10, output_size=3)
print("Model architecture:")
print(model)
print()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

# Test forward pass
x = torch.randn(5, 4)
output = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

Model architecture:
SimpleNet(
  (linear1): Linear(in_features=4, out_features=10, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=10, out_features=3, bias=True)
)

Total parameters: 83
Trainable parameters: 83
Input shape: torch.Size([5, 4])
Output shape: torch.Size([5, 3])


In [7]:
# Test training loop with dummy data
def create_dummy_data(num_samples=1000):
    X = torch.randn(num_samples, 4)
    y = (X[:, 0] + X[:, 1] > 0).long()
    return X, y

X_train, y_train = create_dummy_data(800)
X_val, y_val = create_dummy_data(200)

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Validation labels shape: {y_val.shape}")
print(f"Unique labels: {torch.unique(y_train)}")

# Define model for binary classification
model = nn.Sequential(
    nn.Linear(4, 10),
    nn.ReLU(),
    nn.Linear(10, 2)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val)
    
    train_losses.append(loss.item())
    val_losses.append(val_loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}')

print("Training completed!")

# Evaluate model
model.eval()
with torch.no_grad():
    train_outputs = model(X_train)
    train_predictions = torch.argmax(train_outputs, dim=1)
    train_accuracy = (train_predictions == y_train).float().mean()
    
    val_outputs = model(X_val)
    val_predictions = torch.argmax(val_outputs, dim=1)
    val_accuracy = (val_predictions == y_val).float().mean()

print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

Training data shape: torch.Size([800, 4])
Training labels shape: torch.Size([800])
Validation data shape: torch.Size([200, 4])
Validation labels shape: torch.Size([200])
Unique labels: tensor([0, 1])
Epoch [20/100], Train Loss: 0.2714, Val Loss: 0.2781
Epoch [40/100], Train Loss: 0.2201, Val Loss: 0.2332
Epoch [60/100], Train Loss: 0.1981, Val Loss: 0.2128
Epoch [80/100], Train Loss: 0.1855, Val Loss: 0.2009
Epoch [100/100], Train Loss: 0.1773, Val Loss: 0.1927
Training completed!
Training Accuracy: 0.9275 (92.75%)
Validation Accuracy: 0.9200 (92.00%)


In [8]:
# Test Iris dataset loading and preprocessing
iris = load_iris()
X, y = iris.data, iris.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Feature names: {iris.feature_names}")
print(f"Class names: {iris.target_names}")
print()

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

print(f"Training set size: {X_train_tensor.shape[0]}")
print(f"Test set size: {X_test_tensor.shape[0]}")

Dataset shape: (150, 4)
Number of classes: 3
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Class names: ['setosa' 'versicolor' 'virginica']

Training set size: 120
Test set size: 30


In [9]:
# Define Iris classification model
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 3)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

iris_model = IrisNet()
print("Iris Classification Model:")
print(iris_model)
print()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(iris_model.parameters(), lr=0.01)

print(f"Total parameters: {sum(p.numel() for p in iris_model.parameters())}")

Iris Classification Model:
IrisNet(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=3, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)

Total parameters: 195


In [10]:
# Train Iris model
num_epochs = 200
train_losses = []
train_accuracies = []

iris_model.train()
for epoch in range(num_epochs):
    outputs = iris_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        predicted = torch.argmax(outputs, dim=1)
        accuracy = (predicted == y_train_tensor).float().mean()
    
    train_losses.append(loss.item())
    train_accuracies.append(accuracy.item())
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}')

print("\nTraining completed!")

Epoch [50/200], Loss: 0.4966, Accuracy: 0.8417
Epoch [100/200], Loss: 0.2751, Accuracy: 0.9583
Epoch [150/200], Loss: 0.1751, Accuracy: 0.9750
Epoch [200/200], Loss: 0.1226, Accuracy: 0.9917

Training completed!


In [11]:
# Evaluate on test set
iris_model.eval()
with torch.no_grad():
    test_outputs = iris_model(X_test_tensor)
    test_predictions = torch.argmax(test_outputs, dim=1)
    test_accuracy = (test_predictions == y_test_tensor).float().mean()
    
    test_probabilities = torch.softmax(test_outputs, dim=1)

print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print()

# Show predictions
print("Sample predictions:")
print("Actual -> Predicted (Probabilities)")
for i in range(min(10, len(y_test))):
    actual = iris.target_names[y_test[i]]
    predicted = iris.target_names[test_predictions[i]]
    probs = test_probabilities[i].numpy()
    print(f"{actual:10} -> {predicted:10} {probs}")

Test Accuracy: 1.0000 (100.00%)

Sample predictions:
Actual -> Predicted (Probabilities)
virginica  -> virginica  [7.1037e-07 3.8832e-04 9.9961e-01]
versicolor -> versicolor [8.9851e-04 9.9884e-01 2.6071e-04]
setosa     -> setosa     [9.9999e-01 8.1346e-06 1.1969e-06]
virginica  -> virginica  [9.8671e-06 2.1003e-03 9.9789e-01]
setosa     -> setosa     [9.9999e-01 6.8639e-06 2.6851e-06]
virginica  -> virginica  [2.6423e-07 7.3830e-05 9.9992e-01]
setosa     -> setosa     [1.0000e+00 2.2244e-07 1.1415e-07]
versicolor -> versicolor [3.5894e-05 9.9986e-01 1.0648e-04]
versicolor -> versicolor [1.7043e-04 9.9972e-01 1.0933e-04]
versicolor -> versicolor [7.5126e-05 9.9983e-01 9.1618e-05]


In [12]:
# Test prediction function
def predict_iris_class(model, scaler, features):
    model.eval()
    with torch.no_grad():
        features_scaled = scaler.transform([features])
        features_tensor = torch.FloatTensor(features_scaled)
        
        output = model(features_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = torch.argmax(output, dim=1)
        
        return predicted_class.item(), probabilities.numpy()[0]

# Test examples
examples = [
    [5.1, 3.5, 1.4, 0.2],  # Typical setosa
    [6.2, 2.8, 4.8, 1.8],  # Typical versicolor
    [7.2, 3.0, 5.8, 1.6],  # Typical virginica
]

print("Predictions for new samples:")
print("Features [SL, SW, PL, PW] -> Prediction (Probabilities)")
for features in examples:
    pred_class, probs = predict_iris_class(iris_model, scaler, features)
    class_name = iris.target_names[pred_class]
    print(f"{features} -> {class_name} {probs}")

Predictions for new samples:
Features [SL, SW, PL, PW] -> Prediction (Probabilities)
[5.1, 3.5, 1.4, 0.2] -> setosa [1.0000000e+00 5.0117513e-08 6.2067962e-09]
[6.2, 2.8, 4.8, 1.8] -> virginica [2.3162295e-05 3.0952704e-01 6.9024128e-01]
[7.2, 3.0, 5.8, 1.6] -> virginica [3.6033079e-06 5.5778190e-02 9.4421816e-01]
